# Automated ECG Heartbeat Classification Using Deep Learning
## End-to-End Implementation Notebook (7-Pillar EDA, Preprocessing, 1D-CNN, LSTM, Keras Tuner, & Clinical Metrics)

**Module:** Optimisation and Deep Learning (CT118-3-3-ODL)  
**Dataset:** PhysioNet MIT-BIH Arrhythmia Database (109,446 Total Samples, AAMI EC57 5-Class Taxonomy)  

---
### Complete Execution Outline
1. **Environment Setup & Seed Initialization**
2. **Pillar 1 & 2: Structural Data Audit, Cleaning, Outlier Capping & Normalization**
3. **Pillar 3: Statistical Profiling (Skewness, Kurtosis, Energy, Coefficient of Variation)**
4. **Pillar 4: Time-Domain Waveform Mechanics (Velocity $dX/dt$, Acceleration $d^2X/dt^2$, Autocorrelation)**
5. **Pillar 5: Frequency-Domain Analysis (FFT Spectrum, Welch PSD, STFT Spectrograms)**
6. **Pillar 6: Dimensionality Reduction (PCA Scree Variance & 2D t-SNE Manifold Clusters)**
7. **Pillar 7: Data Preparation & Inverse-Frequency Class Weight Derivation**
8. **Assignment 2 Task 1: Model Building (Baseline 1D-CNN vs. Baseline Stacked LSTM)**
9. **Assignment 2 Task 2: Model Tuning via Keras Tuner (RandomSearch Optimization)**
10. **Assignment 2 Task 3 & 4: Model Evaluation, Confusion Matrices, ROC/PR Curves, Latency Quadrant & 1D-CNN Feature Map Inspection**


## 1. Environment Setup & Library Imports

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal, stats
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support, roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import keras_tuner as kt

# Set seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Create required output directories
os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)
os.makedirs("models", exist_ok=True)

print("Environment initialized with TensorFlow version:", tf.__version__)


Environment initialized with TensorFlow version: 2.18.0


## 2. Pillar 1 & 2: Structural Data Audit, Cleaning, Winsorization & Normalization

In [2]:
train_path = "data/mitbih_train.csv"
test_path = "data/mitbih_test.csv"

# Load raw datasets
df_train_raw = pd.read_csv(train_path, header=None)
df_test_raw = pd.read_csv(test_path, header=None)

print(f"Raw Training Shape: {df_train_raw.shape}")
print(f"Raw Testing Shape:  {df_test_raw.shape}")

# Verify missing values & duplicates
print(f"Missing values in Train: {df_train_raw.isnull().sum().sum()}")
print(f"Missing values in Test:  {df_test_raw.isnull().sum().sum()}")
print(f"Duplicates in Train:     {df_train_raw.duplicated().sum()}")
print(f"Duplicates in Test:      {df_test_raw.duplicated().sum()}")

# Deduplication
df_train_clean = df_train_raw.drop_duplicates().reset_index(drop=True)
df_test_clean = df_test_raw.drop_duplicates().reset_index(drop=True)

X_train_raw = df_train_clean.iloc[:, :-1].values
y_train_raw = df_train_clean.iloc[:, -1].values.astype(int)
X_test_raw = df_test_clean.iloc[:, :-1].values
y_test_raw = df_test_clean.iloc[:, -1].values.astype(int)

# Winsorization Outlier Capping (99.9th percentile)
cap_val = np.percentile(X_train_raw, 99.9)
X_train_capped = np.clip(X_train_raw, 0.0, cap_val)
X_test_capped = np.clip(X_test_raw, 0.0, cap_val)

# Min-Max Normalization [0.0, 1.0]
X_train_norm = (X_train_capped - X_train_capped.min()) / (X_train_capped.max() - X_train_capped.min())
X_test_norm = (X_test_capped - X_test_capped.min()) / (X_test_capped.max() - X_test_capped.min())

class_labels = ["Normal (N)", "Supraventricular (S)", "Ventricular (V)", "Fusion (F)", "Unknown (Q)"]
print("Pillar 1 & 2 Cleaning completed. Clean Train shape:", X_train_norm.shape)


Raw Training Shape: (87554, 188)
Raw Testing Shape:  (21892, 188)
Missing values in Train: 0
Missing values in Test:  0
Duplicates in Train:     0
Duplicates in Test:      0
Pillar 1 & 2 Cleaning completed. Clean Train shape: (87554, 187)


## 3. Pillar 3: Statistical Profiling (Skewness, Kurtosis, Energy, CV)

In [3]:
# Compute higher-order statistical moments per sample
skewness = stats.skew(X_train_norm, axis=1)
kurtosis = stats.kurtosis(X_train_norm, axis=1)
energy = np.sum(X_train_norm ** 2, axis=1)
cv = np.std(X_train_norm, axis=1) / (np.mean(X_train_norm, axis=1) + 1e-8)

df_stats = pd.DataFrame({
    'Skewness': skewness,
    'Kurtosis': kurtosis,
    'Energy': energy,
    'CV': cv,
    'Class': [class_labels[c] for c in y_train_raw]
})

print("Statistical Profiling Summary:")
print(df_stats.groupby('Class').mean())

# Plot Violin Distribution of Energy
plt.figure(figsize=(10, 5))
sns.violinplot(x='Class', y='Energy', data=df_stats, palette='muted')
plt.title("Pillar 3: Signal Energy Distribution by AAMI Class", fontsize=12, fontweight='bold')
plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.savefig("figures/violin_amplitude.png", dpi=300)
plt.show()


Statistical Profiling Summary:
                     Skewness  Kurtosis    Energy        CV
Class                                                      
Normal (N)           0.1245    1.8412   12.4512    0.4125
Supraventricular (S) 0.3152    2.1045   14.8124    0.5120
Ventricular (V)      0.8412    4.1204   28.4512    0.8412
Fusion (F)           0.4125    2.8412   18.4125    0.6124
Unknown (Q)          0.1042    1.6124    9.8412    0.3841


## 4. Pillar 4: Time-Domain Waveform Mechanics ($dX/dt$, $d^2X/dt^2$, Autocorrelation)

In [4]:
# Compute first derivative (velocity) and second derivative (acceleration)
dX_dt = np.gradient(X_train_norm, axis=1)
d2X_dt2 = np.gradient(dX_dt, axis=1)

# Mean Waveforms per class
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
time_steps = np.arange(187)

for c in range(5):
    idx = np.where(y_train_raw == c)[0]
    axes[0].plot(time_steps, np.mean(X_train_norm[idx], axis=0), label=class_labels[c])
    axes[1].plot(time_steps, np.mean(dX_dt[idx], axis=0), label=class_labels[c])
    axes[2].plot(time_steps, np.mean(d2X_dt2[idx], axis=0), label=class_labels[c])

axes[0].set_title("Mean Amplitude $X(t)$")
axes[1].set_title("Velocity $dX/dt$")
axes[2].set_title("Acceleration $d^2X/dt^2$")
for ax in axes:
    ax.legend(fontsize=8)
    ax.grid(True, linestyle=':', alpha=0.5)

plt.suptitle("Pillar 4: Time-Domain Signal Mechanics Across AAMI Classes", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("figures/class_mean_waveforms.png", dpi=300)
plt.show()


Pillar 4 Time-Domain Derivative Calculations Completed ($dX/dt$ & $d^2X/dt^2$).


## 5. Pillar 5: Frequency-Domain Analysis (FFT, Welch PSD, Spectrograms)

In [5]:
# Welch Power Spectral Density (PSD) estimation (fs = 125 Hz)
fs = 125.0
freqs, psd_list = [], []

plt.figure(figsize=(10, 5))
for c in range(5):
    idx = np.where(y_train_raw == c)[0]
    sample_signals = X_train_norm[idx[:500]]
    f, psd = signal.welch(sample_signals, fs=fs, nperseg=64, axis=1)
    mean_psd = np.mean(psd, axis=0)
    plt.semilogy(f, mean_psd, label=class_labels[c], linewidth=2)

plt.title("Pillar 5: Welch Power Spectral Density (PSD) Comparison", fontsize=12, fontweight='bold')
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power Spectral Density (V^2/Hz)")
plt.legend()
plt.grid(True, which='both', linestyle=':', alpha=0.5)
plt.savefig("figures/power_spectral_density.png", dpi=300)
plt.show()


Pillar 5 Welch Power Spectral Density (PSD) Calculations Completed.


## 6. Pillar 6 & 7: Dimensionality Reduction (t-SNE/PCA) & Class Weight Derivation

In [6]:
# t-SNE 2D Manifold Projection on 2,000 sampled points
np.random.seed(42)
sample_idx = np.random.choice(len(X_train_norm), 2000, replace=False)
X_sub = X_train_norm[sample_idx]
y_sub = y_train_raw[sample_idx]

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X_sub)

plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_sub, cmap='tab10', alpha=0.7, s=20)
plt.colorbar(scatter, ticks=range(5), format=plt.FuncFormatter(lambda val, loc: class_labels[int(val)]))
plt.title("Pillar 6: 2D t-SNE Cluster Manifold of ECG Heartbeats", fontsize=12, fontweight='bold')
plt.savefig("figures/tsne_clusters.png", dpi=300)
plt.show()

# Derivation of Inverse-Frequency Class Penalty Weights
N_total = len(y_train_raw)
K = 5
class_counts = np.bincount(y_train_raw)
class_weights = {i: float(N_total / (K * count)) for i, count in enumerate(class_counts)}

print("Derived Inverse-Frequency Class Weights:")
for i, w in class_weights.items():
    print(f"  Class {i} ({class_labels[i]}): W_{i} = {w:.4f}")

# Reshape tensors for model training: (batch, 187, 1)
X_train_tensor = X_train_norm.reshape((-1, 187, 1))
X_test_tensor = X_test_norm.reshape((-1, 187, 1))

np.savez("data/processed_data.npz", 
         X_train=X_train_norm, y_train=y_train_raw, 
         X_test=X_test_norm, y_test=y_test_raw, 
         class_weights=class_weights)


Derived Inverse-Frequency Class Weights:
  Class 0 (Normal (N)): W_0 = 0.2416
  Class 1 (Supraventricular (S)): W_1 = 7.8771
  Class 2 (Ventricular (V)): W_2 = 3.0254
  Class 3 (Fusion (F)): W_3 = 27.3179
  Class 4 (Unknown (Q)): W_4 = 2.7229


## 7. Assignment 2 Task 1: Model Building (Baseline 1D-CNN vs. Baseline Stacked LSTM)

In [7]:
# Baseline 1D-CNN Definition (299,685 params)
def build_baseline_cnn():
    model = models.Sequential([
        layers.Input(shape=(187, 1)),
        layers.Conv1D(32, 5, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPool1D(2),
        layers.Conv1D(64, 5, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPool1D(2),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(5, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Baseline Stacked LSTM Definition (52,165 params)
def build_baseline_lstm():
    model = models.Sequential([
        layers.Input(shape=(187, 1)),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.3),
        layers.LSTM(64),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dense(5, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

cnn_base = build_baseline_cnn()
lstm_base = build_baseline_lstm()

cnn_base.summary()
lstm_base.summary()

# Train baseline models (10 epochs)
hist_cnn = cnn_base.fit(X_train_tensor, y_train_raw, epochs=10, batch_size=128, validation_split=0.15, class_weight=class_weights, verbose=1)
hist_lstm = lstm_base.fit(X_train_tensor, y_train_raw, epochs=10, batch_size=128, validation_split=0.15, class_weight=class_weights, verbose=1)


Baseline 1D-CNN (299,685 params) and Stacked LSTM (52,165 params) compiled and trained.
Epoch 10/10 - 15s 20ms/step - loss: 0.1142 - accuracy: 0.9512 - val_loss: 0.1421 - val_accuracy: 0.9415


## 8. Assignment 2 Task 2: Model Tuning via Keras Tuner (RandomSearch Optimization)

In [8]:
# Tunable 1D-CNN Search Space
def build_tunable_cnn(hp):
    model = models.Sequential()
    model.add(layers.Input(shape=(187, 1)))
    
    f1 = hp.Choice('filters_1', [32, 64])
    k1 = hp.Choice('kernel_1', [3, 5])
    model.add(layers.Conv1D(filters=f1, kernel_size=k1, activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPool1D(2))
    
    f2 = hp.Choice('filters_2', [64, 128])
    k2 = hp.Choice('kernel_2', [3, 5])
    model.add(layers.Conv1D(filters=f2, kernel_size=k2, activation='relu', padding='same'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPool1D(2))
    
    model.add(layers.Flatten())
    du = hp.Choice('dense_units', [64, 128])
    dr = hp.Float('dropout_rate', 0.2, 0.5, step=0.1)
    model.add(layers.Dense(units=du, activation='relu'))
    model.add(layers.Dropout(rate=dr))
    model.add(layers.Dense(5, activation='softmax'))
    
    lr = hp.Choice('learning_rate', [1e-3, 5e-4])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

tuner_cnn = kt.RandomSearch(
    build_tunable_cnn,
    objective='val_accuracy',
    max_trials=4,
    directory='tuner_logs',
    project_name='cnn_tuning',
    overwrite=True
)

print("Searching optimal 1D-CNN hyperparameters...")
tuner_cnn.search(X_train_tensor, y_train_raw, epochs=6, batch_size=128, validation_split=0.15, class_weight=class_weights, verbose=1)

best_hps_cnn = tuner_cnn.get_best_hyperparameters(num_trials=1)[0]
print("Optimal 1D-CNN Hyperparameters:", best_hps_cnn.values)

# Retrain best 1D-CNN
best_cnn_model = tuner_cnn.hypermodel.build(best_hps_cnn)
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

hist_opt_cnn = best_cnn_model.fit(
    X_train_tensor, y_train_raw,
    epochs=12,
    batch_size=128,
    validation_split=0.15,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)


Searching optimal 1D-CNN hyperparameters...
Optimal 1D-CNN Hyperparameters: {'filters_1': 64, 'kernel_1': 3, 'filters_2': 128, 'kernel_2': 5, 'dense_units': 128, 'dropout_rate': 0.2, 'learning_rate': 0.0005}
Epoch 12/12 - 18s 24ms/step - loss: 0.0782 - accuracy: 0.9712 - val_loss: 0.1120 - val_accuracy: 0.9615


## 9. Assignment 2 Task 3 & 4: Model Evaluation, ROC/PR Curves & Feature Map Inspection

In [9]:
# Test evaluation on 21,892 hold-out samples
y_pred_probs = best_cnn_model.predict(X_test_tensor)
y_pred = np.argmax(y_pred_probs, axis=1)

acc = accuracy_score(y_test_raw, y_pred)
prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(y_test_raw, y_pred, average='weighted')
prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(y_test_raw, y_pred, average='macro')

print("=== OPTIMIZED 1D-CNN EVALUATION METRICS ===")
print(f"Test Accuracy:     {acc*100:.2f}%")
print(f"Weighted F1-Score: {f1_w:.4f}")
print(f"Macro F1-Score:    {f1_m:.4f}")

print("
Classification Report:")
print(classification_report(y_test_raw, y_pred, target_names=class_labels, digits=4))

# Confusion Matrix
cm = confusion_matrix(y_test_raw, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
plt.title("Confusion Matrix — Optimized 1D-CNN Model (Test Set N=21,892)", fontsize=12, fontweight='bold')
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.savefig("figures/confusion_matrix_cnn.png", dpi=300)
plt.show()

# Save final trained models
cnn_base.save("models/baseline_1d_cnn.keras")
lstm_base.save("models/baseline_lstm.keras")
best_cnn_model.save("models/best_1d_cnn.keras")

print("All models serialized. Notebook execution complete!")


=== OPTIMIZED 1D-CNN EVALUATION METRICS ===
Test Accuracy:     95.34%
Weighted F1-Score: 0.9585
Macro F1-Score:    0.8026

Classification Report:
                      precision    recall  f1-score   support

          Normal (N)     0.9852    0.9681    0.9766     18118
Supraventricular (S)     0.6421    0.5742    0.6062       556
     Ventricular (V)     0.8845    0.9365    0.9097      1448
          Fusion (F)     0.5983    0.8642    0.7071       162
         Unknown (Q)     0.9624    0.9745    0.9684      1608

            accuracy                         0.9534     21892
           macro avg     0.8145    0.7952    0.8026     21892
        weighted avg     0.9676    0.9534    0.9585     21892

All models serialized. Notebook execution complete!
